# Chapter 8: Supervised Learning — Linear, Ridge, Lasso & Logistic Regression
### Continuous Regularization & Binary Classification Underwriting

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/v893979/Modern_Machine_Learning_and_Data_Science_Engineering/blob/main/src/ch08_linear_and_logistic_regression.ipynb)

**Modern Machine Learning & Data Science Engineering**  
Author: *Vaibhav Rastogi*


### 0. Environment Setup & Cloud Datasets Download
Downloads `university_admissions.csv` and `credit_risk_dataset.csv` if missing.

In [ ]:
import os, urllib.request
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso, LogisticRegression
from sklearn.metrics import r2_score, mean_squared_error, classification_report, roc_auc_score

data_dir = 'data'
os.makedirs(data_dir, exist_ok=True)
base_url = 'https://raw.githubusercontent.com/v893979/Modern_Machine_Learning_and_Data_Science_Engineering/main/data/'

for fname in ['university_admissions.csv', 'credit_risk_dataset.csv']:
    fpath = os.path.join(data_dir, fname)
    if not os.path.exists(fpath):
        urllib.request.urlretrieve(base_url + fname, fpath)
print('All datasets verified ready in data/')

### 1. Continuous Regression: OLS vs. Ridge (L2) vs. Lasso (L1)
We compare Standard OLS, Ridge ($\alpha=10$), and Lasso ($\alpha=0.03$) on standardized features.

In [ ]:
df_adm = pd.read_csv('data/university_admissions.csv')
features = ['gre_score', 'toefl_score', 'univ_rating', 'sop', 'lor', 'cgpa', 'research']
X = df_adm[features]
y = df_adm['chance_of_admit']

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.20, random_state=42)
scaler = StandardScaler()
X_tr_sc = scaler.fit_transform(X_tr)
X_te_sc = scaler.transform(X_te)

ols = LinearRegression().fit(X_tr_sc, y_tr)
ridge = Ridge(alpha=10.0).fit(X_tr_sc, y_tr)
lasso = Lasso(alpha=0.03).fit(X_tr_sc, y_tr)

print(f"{'Feature':<14} {'Standard OLS':>14} {'Ridge (L2)':>14} {'Lasso (L1)':>14}")
print('-' * 60)
for feat, o_w, r_w, l_w in zip(features, ols.coef_, ridge.coef_, lasso.coef_):
    print(f"{feat:<14} {o_w:>14.4f} {r_w:>14.4f} {l_w:>14.4f}")
print('-' * 60)
print(f"Holdout R^2 Score: OLS={r2_score(y_te, ols.predict(X_te_sc)):.4f} | Ridge={r2_score(y_te, ridge.predict(X_te_sc)):.4f} | Lasso={r2_score(y_te, lasso.predict(X_te_sc)):.4f}")

### 2. Binary Classification: Logistic Regression Credit Baseline

In [ ]:
df_credit = pd.read_csv('data/credit_risk_dataset.csv')
credit_feats = ['person_age', 'person_income', 'person_emp_length', 'loan_amnt', 'loan_int_rate', 'loan_percent_income']
Xc = df_credit[credit_feats]
yc = df_credit['loan_status']

Xc_tr, Xc_te, yc_tr, yc_te = train_test_split(Xc, yc, test_size=0.25, random_state=42, stratify=yc)
sc_c = StandardScaler()
Xc_tr_sc = sc_c.fit_transform(Xc_tr)
Xc_te_sc = sc_c.transform(Xc_te)

log_reg = LogisticRegression(solver='lbfgs', max_iter=1000, random_state=42)
log_reg.fit(Xc_tr_sc, yc_tr)

preds = log_reg.predict(Xc_te_sc)
probs = log_reg.predict_proba(Xc_te_sc)[:, 1]

print(f"Logistic Regression ROC-AUC: {roc_auc_score(yc_te, probs):.4f}\n")
print(classification_report(yc_te, preds, digits=4))